# SaaS Revenue Movement Analysis

## Objective
Analyze customer revenue movements and identify:
- New MRR
- Expansion MRR
- Contraction MRR
- Churn MRR

## Input
- cleaned_subscription_data.csv

## Output
- final_mrr_table.csv
- monthly_expansion.csv
- monthly_contraction.csv
- monthly_churn.csv

## Business Goal
Understand how customer revenue changes over time and identify revenue growth and revenue leakage drivers.

In [112]:
import pandas as pd
import numpy as np

In [113]:
subscription = pd.read_csv('cleaned_subscription_data.csv')

In [114]:
subscription.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   subscription_id    5000 non-null   object
 1   account_id         5000 non-null   object
 2   start_date         5000 non-null   object
 3   end_date           486 non-null    object
 4   plan_tier          5000 non-null   object
 5   seats              5000 non-null   int64 
 6   mrr_amount         5000 non-null   int64 
 7   arr_amount         5000 non-null   int64 
 8   is_trial           5000 non-null   bool  
 9   upgrade_flag       5000 non-null   bool  
 10  downgrade_flag     5000 non-null   bool  
 11  churn_flag         5000 non-null   bool  
 12  billing_frequency  5000 non-null   object
 13  auto_renew_flag    5000 non-null   bool  
 14  year_and_month     5000 non-null   object
 15  month              5000 non-null   object
 16  year               5000 non-null   int64 


# Active customers

In [115]:
total_active_cust = subscription[
subscription['status'] == 'Active'
]

In [116]:
total_active_cust.groupby('status')['account_id'].nunique()

status
Active    500
Name: account_id, dtype: int64

In [117]:
total_active_cust.to_csv('total_active_cust.csv',index= False)

## Build Customer-Month Revenue Table

Aggregate revenue at the customer-month level.

This creates a single monthly revenue record per customer, which becomes the foundation for movement analysis.

In [118]:
cust_month_rev = subscription.groupby(['year_and_month','account_id'])['mrr_amount'].sum().reset_index()
cust_month_rev = cust_month_rev.sort_values(['account_id','year_and_month'])
cust_month_rev

,year_and_month,account_id,mrr_amount
338,2023-11,A-00bed1,1159
877,2024-04,A-00bed1,1372
1014,2024-05,A-00bed1,15323
1350,2024-07,A-00bed1,6944
1567,2024-08,A-00bed1,3136
...,...,...,...
1778,2024-08,A-ffdfd5,570
2026,2024-09,A-ffdfd5,4508
2299,2024-10,A-ffdfd5,2189
2599,2024-11,A-ffdfd5,5970


## Calculate Previous Month Revenue

We calculate each customer's revenue from the previous month.

This allows us to compare current revenue against historical revenue and classify customer movements.

In [119]:
cust_month_rev['previous_mrr'] = cust_month_rev.groupby('account_id')['mrr_amount'].shift(periods = 1)
cust_month_rev


,year_and_month,account_id,mrr_amount,previous_mrr
338,2023-11,A-00bed1,1159,NaN
877,2024-04,A-00bed1,1372,1159.0
1014,2024-05,A-00bed1,15323,1372.0
1350,2024-07,A-00bed1,6944,15323.0
1567,2024-08,A-00bed1,3136,6944.0
...,...,...,...,...
1778,2024-08,A-ffdfd5,570,399.0
2026,2024-09,A-ffdfd5,4508,570.0
2299,2024-10,A-ffdfd5,2189,4508.0
2599,2024-11,A-ffdfd5,5970,2189.0


In [120]:
cust_month_rev[cust_month_rev['account_id'] == 'A-00bed1']

,year_and_month,account_id,mrr_amount,previous_mrr
338,2023-11,A-00bed1,1159,NaN
877,2024-04,A-00bed1,1372,1159.0
1014,2024-05,A-00bed1,15323,1372.0
1350,2024-07,A-00bed1,6944,15323.0
1567,2024-08,A-00bed1,3136,6944.0
1779,2024-09,A-00bed1,5572,3136.0


## Classify Revenue Movements

Each customer-month is classified into:

- New
- Expansion
- Contraction
- Churn
- No Change

This helps identify revenue growth and revenue loss drivers.

In [121]:
# movement type
conditions = [
    (((cust_month_rev['previous_mrr'] == 0) | (cust_month_rev['previous_mrr'].isna())) & (cust_month_rev['mrr_amount'] > 0)),
    (cust_month_rev['mrr_amount'] > cust_month_rev['previous_mrr']),
    ((cust_month_rev['previous_mrr'] > 0) & (cust_month_rev['mrr_amount'] == 0)),
    (cust_month_rev['mrr_amount'] < cust_month_rev['previous_mrr']),
    (cust_month_rev['mrr_amount'] == cust_month_rev['previous_mrr'])
]

choice = ['New','Expansion','Churn','Contraction','NoChange']

cust_month_rev['movement_type'] = np.select(conditions,choice,default = 'Unknown')

In [122]:
cust_month_rev

,year_and_month,account_id,mrr_amount,previous_mrr,movement_type
338,2023-11,A-00bed1,1159,NaN,New
877,2024-04,A-00bed1,1372,1159.0,Expansion
1014,2024-05,A-00bed1,15323,1372.0,Expansion
1350,2024-07,A-00bed1,6944,15323.0,Contraction
1567,2024-08,A-00bed1,3136,6944.0,Contraction
...,...,...,...,...,...
1778,2024-08,A-ffdfd5,570,399.0,Expansion
2026,2024-09,A-ffdfd5,4508,570.0,Expansion
2299,2024-10,A-ffdfd5,2189,4508.0,Contraction
2599,2024-11,A-ffdfd5,5970,2189.0,Expansion


In [123]:
# checking
cust_month_rev[cust_month_rev['account_id'] == 'A-3c1a3f']

,year_and_month,account_id,mrr_amount,previous_mrr,movement_type
172,2023-08,A-3c1a3f,0,NaN,Unknown
444,2023-12,A-3c1a3f,8557,0.0,New
761,2024-03,A-3c1a3f,76,8557.0,Contraction
913,2024-04,A-3c1a3f,398,76.0,Expansion
1053,2024-05,A-3c1a3f,931,398.0,Expansion
1829,2024-09,A-3c1a3f,597,931.0,Contraction
2089,2024-10,A-3c1a3f,4350,597.0,Expansion


In [124]:
cust_month_rev.groupby('movement_type')['account_id'].count().reset_index()

,movement_type,account_id
0,Churn,222
1,Contraction,865
2,Expansion,953
3,New,678
4,NoChange,158
5,Unknown,59


## Create Customer-Month Matrix

Generate all customer × month combinations.

This prevents missing months from being ignored and enables churn detection.

In [125]:
cust_month_metrix = pd.MultiIndex.from_product([subscription['account_id'].unique(),subscription['year_and_month'].unique()],names = ['account_id','year_and_month'])
cust_month_metrix

MultiIndex([('A-3c1a3f', '2023-12'),
            ('A-3c1a3f', '2024-06'),
            ('A-3c1a3f', '2024-11'),
            ('A-3c1a3f', '2024-01'),
            ('A-3c1a3f', '2024-08'),
            ('A-3c1a3f', '2024-12'),
            ('A-3c1a3f', '2024-10'),
            ('A-3c1a3f', '2024-02'),
            ('A-3c1a3f', '2023-09'),
            ('A-3c1a3f', '2024-07'),
            ...
            ('A-751bd4', '2023-06'),
            ('A-751bd4', '2023-05'),
            ('A-751bd4', '2024-05'),
            ('A-751bd4', '2023-02'),
            ('A-751bd4', '2023-10'),
            ('A-751bd4', '2023-08'),
            ('A-751bd4', '2023-04'),
            ('A-751bd4', '2023-11'),
            ('A-751bd4', '2023-03'),
            ('A-751bd4', '2023-01')],
           names=['account_id', 'year_and_month'], length=12000)

In [126]:
cust_month_metrix.shape

(12000,)

In [127]:
cust_month_metrix = cust_month_metrix.to_frame(index = False)

In [128]:
cust_month_metrix.head()

,account_id,year_and_month
0,A-3c1a3f,2023-12
1,A-3c1a3f,2024-06
2,A-3c1a3f,2024-11
3,A-3c1a3f,2024-01
4,A-3c1a3f,2024-08


In [129]:
# merging cust_month_rev and cust_month_metrix

final_df = pd.merge(cust_month_metrix,cust_month_rev,on=['account_id', 'year_and_month'],how = 'left')
final_df

,account_id,year_and_month,mrr_amount,previous_mrr,movement_type
0,A-3c1a3f,2023-12,8557.0,0.0,New
1,A-3c1a3f,2024-06,NaN,NaN,NaN
2,A-3c1a3f,2024-11,NaN,NaN,NaN
3,A-3c1a3f,2024-01,NaN,NaN,NaN
4,A-3c1a3f,2024-08,NaN,NaN,NaN
...,...,...,...,...,...
11995,A-751bd4,2023-08,NaN,NaN,NaN
11996,A-751bd4,2023-04,NaN,NaN,NaN
11997,A-751bd4,2023-11,NaN,NaN,NaN
11998,A-751bd4,2023-03,NaN,NaN,NaN


In [130]:
final_df.shape

(12000, 5)

In [131]:
final_df['mrr_amount'] = final_df['mrr_amount'].fillna(0)

In [132]:
final_df = final_df.drop(columns=['previous_mrr','movement_type'])

In [133]:
final_df = final_df.sort_values(['account_id','year_and_month'])

In [134]:
final_df['previous_mrr'] = final_df.groupby('account_id')['mrr_amount'].shift(periods = 1)

In [135]:
final_df

,account_id,year_and_month,mrr_amount,previous_mrr
2255,A-00bed1,2023-01,0.0,NaN
2249,A-00bed1,2023-02,0.0,0.0
2254,A-00bed1,2023-03,0.0,0.0
2252,A-00bed1,2023-04,0.0,0.0
2247,A-00bed1,2023-05,0.0,0.0
...,...,...,...,...
10276,A-ffdfd5,2024-08,570.0,0.0
10282,A-ffdfd5,2024-09,4508.0,570.0
10278,A-ffdfd5,2024-10,2189.0,4508.0
10274,A-ffdfd5,2024-11,5970.0,2189.0


## Recalculate Revenue Movements

Merge customer-month matrix with revenue data and classify:

- New
- Expansion
- Contraction
- Churn
- Inactive

In [136]:
conditions = [
    (((final_df['previous_mrr'] == 0) | (final_df['previous_mrr'].isna())) & (final_df['mrr_amount'] > 0)),
    ((final_df['mrr_amount'] > final_df['previous_mrr']) & final_df['previous_mrr'] > 0),
    ((final_df['previous_mrr']>0) & (final_df['mrr_amount'] == 0)),
    (final_df['mrr_amount'] < final_df['previous_mrr']),
    (final_df['mrr_amount'] == final_df['previous_mrr']),
    ((final_df['previous_mrr'].isna()) & (final_df['mrr_amount'] == 0))
]

choice = ['New','Expansion','Churn','Contraction','NoChange','Inactive']

final_df['movement_type'] = np.select(conditions,choice,default = 'Unknown')


In [137]:
final_df.to_csv('final_mrr_table.csv',index = False)

In [138]:
final_df['movement_type'].value_counts()

movement_type
NoChange       7820
New            1438
Churn          1126
Expansion       608
Contraction     510
Inactive        498
Name: count, dtype: int64

In [ ]:
monthly_summery = final_df.groupby(['year_and_month','movement_type'])['mrr_amount'].sum().reset_index()
monthly_summery.sort_values(['year_and_month'])

In [ ]:
mrr_summary = final_df.pivot_table(
    index = 'year_and_month',
    columns = 'movement_type',
    values = 'mrr_amount',
    aggfunc = 'sum',
    fill_value = 0
)

In [ ]:
mrr_summary

## Monthly MRR Analysis

In [ ]:
monthly_mrr = subscription.groupby('year_and_month')['mrr_amount'].sum().reset_index()
monthly_mrr

In [ ]:
monthly_mrr.to_csv('monthly_mrr.csv',index= False)

In [ ]:
final_mrr = pd.merge(mrr_summary,mrr_monthly,on='year_and_month',how = 'left')

In [ ]:
final_mrr.head()

## Expansion MRR Analysis

Expansion MRR measures additional revenue generated from existing customers through upgrades, additional seats, or add-on purchases.

In [ ]:
expansion_df = final_df[
    final_df['movement_type'] == 'Expansion'
].copy()

In [ ]:
expansion_df['expansion_amount'] = (
    expansion_df['mrr_amount'] - expansion_df['previous_mrr'] 
)

In [ ]:
expansion_df

In [ ]:
monthly_expansion = expansion_df.groupby('year_and_month')['expansion_amount'].sum().reset_index()
monthly_expansion

In [ ]:
monthly_expansion.to_csv(
    'monthly_expansion.csv',
    index=False
)

### Insight

Expansion MRR identifies revenue growth from existing customers and highlights successful upselling opportunities.

## Contraction MRR Analysis

Contraction MRR measures revenue lost when customers remain active but downgrade plans or reduce usage.

In [ ]:
contraction_df = final_df[
    final_df['movement_type'] == 'Contraction'
].copy()

In [ ]:
contraction_df['contraction_amount'] = (
    contraction_df['previous_mrr']
    - contraction_df['mrr_amount']
)

In [ ]:
monthly_contraction = (
    contraction_df
    .groupby('year_and_month')['contraction_amount']
    .sum()
    .reset_index()
)

In [ ]:
monthly_contraction

In [ ]:
monthly_contraction.to_csv(
    'monthly_contraction.csv',
    index=False
)

### Insight

High contraction MRR may indicate customers are reducing their spending due to budget constraints, reduced product usage, or lower perceived value.

## Churn MRR Analysis

Churn MRR measures revenue lost when customers completely stop paying for the service.

In [ ]:
churn_df = final_df[
    final_df['movement_type'] == 'Churn'
].copy()

In [ ]:
churn_df['churn_amount'] = (
    churn_df['previous_mrr']
)

In [ ]:
monthly_churn = (
    churn_df
    .groupby('year_and_month')['churn_amount']
    .sum()
    .reset_index()
)

In [ ]:
monthly_churn

In [ ]:
monthly_churn.to_csv(
    'monthly_churn.csv',
    index=False
)

### Insight

Churn MRR is the most critical revenue leakage metric because it represents customers who have completely stopped generating recurring revenue.